# VerSe Preprocessing: Quick Sample Test
This notebook tests the 3D to 2D conversion logic on the single sample subject `sub-verse004` to verify visual correctness before running the full pipeline.

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from PIL import Image
from data_utilities import *

# Paths to the single sample subject
SAMPLE_RAW = 'sample/sub-verse004_ct.nii.gz'
SAMPLE_MSK = 'sample/sub-verse004_seg-vert_msk.nii.gz'

def window_image(img, min_hu=-500, max_hu=1300):
    """Apply bone windowing to CT images."""
    img = np.clip(img, min_hu, max_hu)
    img = (img - min_hu) / (max_hu - min_hu) * 255.0
    return img.astype(np.uint8)

def map_semantic_class(inst_id):
    """1=Cervical, 2=Thoracic, 3=Lumbar, 0=Background"""
    if 1 <= inst_id <= 7: return 1
    if (8 <= inst_id <= 19) or (inst_id == 28): return 2
    if 20 <= inst_id <= 25: return 3
    return 0

def get_panoptic_encoding(inst_msk):
    unique_ids = np.unique(inst_msk).astype(int)
    unique_ids = unique_ids[unique_ids > 0]
    pan_img = np.zeros((inst_msk.shape[0], inst_msk.shape[1], 3), dtype=np.uint8)
    for uid in unique_ids:
        mask = (inst_msk == uid)
        pan_img[mask, 0] = uid % 256
        pan_img[mask, 1] = uid // 256
        pan_img[mask, 2] = 0
    return pan_img

print("Libraries and Helper Functions Loaded.")

## 1. Load and Standardize 3D Volume

In [ ]:
img_nii = nib.load(SAMPLE_RAW)
msk_nii = nib.load(SAMPLE_MSK)

# Resample to 1mm isotropic and reorient to Sagittal orientation
im_iso = resample_nib(img_nii, (1,1,1), 3)
ms_iso = resample_nib(msk_nii, (1,1,1), 0)
im_np = reorient_to(im_iso, ('I','P','L')).get_fdata()
ms_np = reorient_to(ms_iso, ('I','P','L')).get_fdata()

print(f"Reoriented 3D Shape: {im_np.shape}")
print(f"Number of sagittal slices available: {im_np.shape[2]}")

## 2. Visualization of Preprocessing Tasks

In [ ]:
# Pick a middle slice where the spine is clearly visible
vis_idx = im_np.shape[2] // 2

img_slice = window_image(im_np[:,:,vis_idx])
inst_slice = ms_np[:,:,vis_idx].astype(np.uint8)
sem_slice = np.vectorize(map_semantic_class)(inst_slice).astype(np.uint8)
pan_slice = get_panoptic_encoding(inst_slice)

fig, ax = plt.subplots(1, 4, figsize=(24, 6))

ax[0].imshow(img_slice, cmap='gray')
ax[0].set_title(f"Bone Windowed Image (Slice {vis_idx})")

ax[1].imshow(sem_slice, cmap='tab10')
ax[1].set_title("Semantic Segmentation (Regions)")

ax[2].imshow(inst_slice, cmap='nipy_spectral')
ax[2].set_title("Instance Segmentation (Vertebrae)")

ax[3].imshow(pan_slice)
ax[3].set_title("Panoptic Encoding (Machine Labels)")

for a in ax: a.axis('off')
plt.tight_layout()
plt.show()

## 3. Slice Quality Check
We only keep slices with enough vertebral pixels to ensure high-quality training data.

In [ ]:
areas = [np.sum(ms_np[:,:,i] > 0) for i in range(im_np.shape[2])]
threshold = 500 # Minimum pixels

plt.figure(figsize=(10, 4))
plt.plot(areas, label='Vertebrae Pixels')
plt.axhline(y=threshold, color='r', linestyle='--', label='Min Quality Threshold')
plt.title("Bone Area per Slice")
plt.xlabel("Slice Index")
plt.ylabel("Pixel Count")
plt.legend()
plt.show()

kept_count = sum(1 for a in areas if a >= threshold)
print(f"Of the {im_np.shape[2]} slices, {kept_count} would be kept for training.")